# Processing MESACLIP data - ocean daily, single ensembles
Ensembles 004-010 only

* **Description**: Reads in MESACLIP Historical Ocean data
* **Creator**: Alice DuVivier
* **Date**: October 2026

In [1]:
import xarray as xr
import numpy as np
import pandas as pd
from glob import glob
from datetime import datetime
import xesmf as xe
import matplotlib.pyplot as plt
import scipy.sparse as sps

import dask
from distributed import Client
from ncar_jobqueue import NCARCluster

## Spin up DASK cluster

In [2]:
# Create our NCAR Cluster - which uses PBSCluster under the hood
cluster = NCARCluster(walltime='12:00:00', cores=1, processes=1, memory='10 GB', 
                      resource_spec='select=1:ncpus=1:mem=10GB',interface='ext')

# Spin up 8 workers
cluster.scale(8)

# Assign the cluster to our Client
client = Client(cluster)

/glade/u/apps/opt/conda/envs/npl-2024a/lib/python3.11/site-packages/dask_jobqueue/core.py:255: FutureWarning: job_extra has been renamed to job_extra_directives. You are still using it (even if only set to []; please also check config files). If you did not set job_extra_directives yet, job_extra will be respected for now, but it will be removed in a future release. If you already set job_extra_directives, job_extra is ignored and you can remove it.
  warnings.warn(warn, FutureWarning)
/glade/u/apps/opt/conda/envs/npl-2024a/lib/python3.11/site-packages/dask_jobqueue/core.py:274: FutureWarning: env_extra has been renamed to job_script_prologue. You are still using it (even if only set to []; please also check config files). If you did not set job_script_prologue yet, env_extra will be respected for now, but it will be removed in a future release. If you already set job_script_prologue, env_extra is ignored and you can remove it.
  warnings.warn(warn, FutureWarning)
/glade/u/apps/opt/con

In [3]:
client
# cluster.close()

Connection method: Cluster object,Cluster type: dask_jobqueue.PBSCluster
Dashboard: /proxy/8787/status,
Dashboard: /proxy/8787/status,Workers: 0
Total threads: 0,Total memory: 0 B
Comm: tcp://128.117.208.177:43035,Workers: 0
Dashboard: /proxy/8787/status,Total threads: 0
Started: Just now,Total memory: 0 B


## Set MESACLIP ensemble information

In [4]:
# Historical
path_base = '/gdex/data/d651007/'
case_base = 'b.e13.BHISTC5.ne120_t12.cesm-ihesp-'
# list 10 members of MESACLIP - last one is the first and named differently
ens_mems = ['sehires38-1850-2005.001', 'hires1.0.30-1920-2005.002', 
            'hires1.0.30-1920-2005.003', 'hires1.0.44-1920-2005.004', 
            'hires1.0.44-1920-2005.005', 'hires1.0.45-1920-2005.006', 
            'hires1.0.46-1920-2005.007', 'hires1.0.46-1920-2005.008',
            'hires1.0.46-1920-2005.009', 'hires1.0.46-1920-2005.010']

nens = len(ens_mems)

In [6]:
# set the groups of years since we have to load them separately
# 1920s, 1930s, 1940s, 1950s, 1960s, 1970s, 1980s, 1990s, 2000s
year_sets = ['192','193','194','195','196',
             '197','198','199','200']

nyears = len(year_sets)

## Load in Ocean Data

In [7]:
# Choose variable to load
var_in = 'SSH_2'

In [8]:
# set output directory
path_out = '/glade/campaign/cgd/ppc/duvivier/arctic_actionable/DATA/mesaclip_all_arctic/daily/SSH/'

In [9]:
# define preprocess function applied to each file as it is opened:
# keep only the variable of interest and time bounds, then crop to Alaskan Arctic Region: 50-80N Lat, 170-230 Lon
def subset(ds):
    ds = ds[[var_in, 'time_bound']]
    return ds.isel(nlat=slice(1700, 2250), nlon=slice(2700, 3300))

In [ ]:
# loop through each ensemble and each set of years

# loop through ensembles
for n in np.arange(0,nens,1)[3:10]:
    # loop through year groupings
    for y in np.arange(0,nyears,1):
        ens_in = ens_mems[n]
        yr_pre = year_sets[y]
        print('loading ensemble: '+ens_in+' for years '+yr_pre+'0s')
        # set case
        case = case_base + ens_in
        # set data directory
        path = path_base+case+'/ocn/proc/tseries/day_1'
        # get list of files
        files = sorted(glob(f'{path}/{case}.pop.h.nday1.{var_in}.{yr_pre}?????-????????.nc'))       
        # load files, subsetting each one as it is opened
        ds_sub = xr.open_mfdataset(files, preprocess=subset, chunks={'time': 365},
                                   data_vars='minimal', coords='minimal', compat='override',
                                   parallel=True, concat_dim='time', combine='nested',
                                   decode_times=True)
        # take an average over the time bounds to get the right time dimension
        ds_sub['time'] = ds_sub.time_bound.compute().mean(dim='d2')

        # save and export data
        # set output file name
        nn = n+1
        if nn < 10:
            ensnum = '00'+str(nn)
        elif nn >= 10:
            ensnum = '010'

        file_out = 'MESACLIP_HIST_'+ensnum+'_'+yr_pre+'0-'+yr_pre+'9'+'_SSH_alaska_daily.nc'
        fout = path_out + file_out
        #print(fout)

        # make output array
        ds_out = ds_sub.rename({var_in: 'SSH'}).load()
        ds_out = ds_out.load()
        # change the attributes
        ds_out.attrs['author'] = 'Alice DuVivier'
        ds_out.attrs['date_processed'] = datetime.now().strftime('%Y-%m-%d')
        ds_out.attrs['contents'] = f'Subset of historical MESACLIP Data'
        # calculate the size of the dataset in GB
        size_gb = ds_out.nbytes / (1024**3)
        print(f"The dataset is approximately {size_gb:.2f} GB.")

        # Export the dataset to NetCDF with all attributes and coordinates
        print('Writing post processed file')
        ds_out.to_netcdf(fout)


loading ensemble: hires1.0.44-1920-2005.004 for years 1920s
The dataset is approximately 4.50 GB.
Writing post processed file
loading ensemble: hires1.0.44-1920-2005.004 for years 1930s
The dataset is approximately 4.50 GB.
Writing post processed file
loading ensemble: hires1.0.44-1920-2005.004 for years 1940s
The dataset is approximately 4.50 GB.
Writing post processed file
loading ensemble: hires1.0.44-1920-2005.004 for years 1950s
The dataset is approximately 4.50 GB.
Writing post processed file
loading ensemble: hires1.0.44-1920-2005.004 for years 1960s
The dataset is approximately 4.50 GB.
Writing post processed file
loading ensemble: hires1.0.44-1920-2005.004 for years 1970s
The dataset is approximately 4.50 GB.
Writing post processed file
loading ensemble: hires1.0.44-1920-2005.004 for years 1980s
The dataset is approximately 4.50 GB.
Writing post processed file
loading ensemble: hires1.0.44-1920-2005.004 for years 1990s
The dataset is approximately 4.50 GB.
Writing post process

In [11]:
print("Done!")
cluster.close()